# Сервис геопланирования

Программный модуль геопланирования, предназначенный для автоматического распределения географических точек (клиентов, магазинов, объектов обслуживания) по дням месяца и формирования кластеров, пригодных для ежедневного обхода.

## Задачи программного решения

- Распределяет точки по количеству требуемых посещений в месяц.
- Формирует кластеры точек, которые можно обслужить за один день.
- Обеспечивает простую визуализацию кластеров и маршрутов (опционально).
- Подготавливает результат в удобной табличной форме.

## Входные данные

Модуль принимает таблицу в формате CSV со следующими полями:

| Поле                | Тип     | Описание                          |
|---------------------|---------|-----------------------------------|
| `point_id`          | string  | Уникальный идентификатор точки    |
| `latitude`          | float   | Широта                            |
| `longitude`         | float   | Долгота                           |
| `visits_per_month`  | integer | Количество посещений в месяц      |

|

# Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from geopy.distance import geodesic
import folium
from folium import plugins
import os
from folium.plugins import GroupedLayerControl
from typing import Dict, List, Tuple
import json
import hashlib
import time
import requests
import webbrowser
import routingpy
import sys
import pyrosm
from pyrosm import OSM
from routingpy import OSRM
import osmnx as ox
import networkx as nx
from scipy.spatial.distance import squareform
import subprocess
import sys
import os
from shapely.geometry import box

import warnings
warnings.filterwarnings('ignore')


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Центр города Нижний Новгород
CITY_CENTER_LAT = 56.3269
CITY_CENTER_LON = 44.0052

# Скорость движения (км/ч) – используется только для расчёта времени в пути
AVG_SPEED = 60

# Папки для кеша и выходных данных
BASE_DIR = r"D:\denis\geolocation"
CACHE_DIR = os.path.join(BASE_DIR, "cache2")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs2")
DATA_PATH = os.path.join(BASE_DIR, "data.csv")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Параметры кластеризации
EPS_KM = 25
MAX_CLUSTER_SIZE = 12
MIN_CLUSTER_SIZE = 8
WORKING_DAYS = 22

print("✅ Библиотеки импортированы, настройки заданы.")
print(f"Центр города: ({CITY_CENTER_LAT}, {CITY_CENTER_LON})")
print(f"Радиус кластеризации: {EPS_KM} км")
print(f"Максимум точек в кластере: {MAX_CLUSTER_SIZE}")
print(f"Рабочих дней: {WORKING_DAYS}")
print(f"Папка кеша: {CACHE_DIR}")
print(f"Папка выходных данных: {OUTPUT_DIR}")

✅ Библиотеки импортированы, настройки заданы.
Центр города: (56.3269, 44.0052)
Радиус кластеризации: 25 км
Максимум точек в кластере: 12
Рабочих дней: 22
Папка кеша: D:\denis\geolocation\cache2
Папка выходных данных: D:\denis\geolocation\outputs2


# 1 Загрузка и валидация данных

In [2]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Файл не найден: {DATA_PATH}")

def load_and_validate_data(filepath):
    df = pd.read_csv(filepath)
    rename_map = {}
    if 'lat' in df.columns and 'latitude' not in df.columns:
        rename_map['lat'] = 'latitude'
    if 'lon' in df.columns and 'longitude' not in df.columns:
        rename_map['lon'] = 'longitude'
    if 'n_visits' in df.columns and 'visits_per_month' not in df.columns:
        rename_map['n_visits'] = 'visits_per_month'
    if rename_map:
        df.rename(columns=rename_map, inplace=True)
        print(f"✅ Переименованы столбцы: {rename_map}")
    required_cols = ['point_id', 'latitude', 'longitude', 'visits_per_month']
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Отсутствуют обязательные колонки: {missing}")
    df.dropna(subset=required_cols, inplace=True)
    df['point_id'] = df['point_id'].astype(str)
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    df['visits_per_month'] = pd.to_numeric(df['visits_per_month'], errors='coerce').astype('Int64')
    df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]
    df = df[df['visits_per_month'] >= 1]
    df.drop_duplicates(subset='point_id', keep='first', inplace=True)
    df['visits_per_month'] = df['visits_per_month'].clip(upper=2)
    return df.reset_index(drop=True)

df_original = load_and_validate_data(DATA_PATH)
print(f"✅ Загружено {len(df_original)} корректных точек.")
df_original.head(10)

✅ Переименованы столбцы: {'lat': 'latitude', 'lon': 'longitude', 'n_visits': 'visits_per_month'}
✅ Загружено 633 корректных точек.


,point_id,manager,latitude,longitude,visits_per_month
0,ID1,0,55.371884,43.846878,1
1,ID2,0,55.358381,43.845061,1
2,ID3,0,55.374786,43.832983,1
3,ID4,0,55.423430,43.813190,1
4,ID5,0,55.381287,43.811452,1
5,ID6,1,55.317955,42.154033,1
6,ID7,1,55.548200,42.064568,1
7,ID8,1,55.317959,42.084073,1
8,ID9,1,55.557407,42.195016,1
9,ID10,1,55.577987,42.033764,1


Реализованно:
* Гибкое переименование столбцов (lat → latitude, n_visits → visits_per_month).
* Проверка на пропуски, выход координат за допустимые диапазоны, удаление дубликатов по point_id.
* Ограничение visits_per_month сверху 2 (логично для задачи, где визиты могут быть кратными).
* Успешно загружено 633 корректных точки.
* Отсутствие проверки на выбросы по координатам (например, точки, явно выпадающие из региона) могло бы улучшить качество кластеризации, но фильтрация по широте/долготе уже отсекает грубые ошибки.

Вывод
* Данные загружены корректно, структура соответствует требованиям. Небольшой объём позволяет применять сложные алгоритмы, но в будущем при масштабировании потребуется оптимизация.

# 2 Загрузка дорожного графа и кеширование расстояний

In [3]:
BASE_DIR = r"D:\denis\geolocation"
CACHE_DIR = os.path.join(BASE_DIR, "cache2")
os.makedirs(CACHE_DIR, exist_ok=True)

GRAPHML_PATH = os.path.join(CACHE_DIR, "volga_graph.graphml")

def clean_graph_for_graphml(graph):
    """
    Удаляет из графа все атрибуты, которые не могут быть сериализованы в GraphML.
    В частности: None, списки, словари, множества и другие нестандартные объекты.
    """
    if graph is None:
        return None
    
    # Очистка атрибутов узлов
    for node, data in list(graph.nodes(data=True)):
        for key in list(data.keys()):
            val = data[key]
            # Удаляем, если значение None, список, словарь, множество или любой другой нестандартный тип
            if val is None or isinstance(val, (list, dict, set, tuple)):
                del data[key]
            # Также можно удалить, если значение не является базовым типом (int, float, str, bool)
            elif not isinstance(val, (int, float, str, bool)):
                del data[key]
    
    # Очистка атрибутов рёбер
    for u, v, key, data in list(graph.edges(data=True, keys=True)):
        for key_attr in list(data.keys()):
            val = data[key_attr]
            if val is None or isinstance(val, (list, dict, set, tuple)):
                del data[key_attr]
            elif not isinstance(val, (int, float, str, bool)):
                del data[key_attr]
    
    return graph

def find_pbf_file(base_dir):
    pattern = os.path.join(base_dir, "*.osm.pbf")
    files = glob.glob(pattern)
    return files[0] if files else None

def build_bbox_from_points(df_points, buffer_km=50):
    if df_points.empty:
        return None
    lat_min, lat_max = df_points['latitude'].min(), df_points['latitude'].max()
    lon_min, lon_max = df_points['longitude'].min(), df_points['longitude'].max()
    buffer_deg = buffer_km / 111.0
    # Возвращаем (south, west, north, east) для использования в pyrosm и OSMnx
    return (lat_min - buffer_deg, lon_min - buffer_deg,
            lat_max + buffer_deg, lon_max + buffer_deg)

def load_graph(df_points=None):
    # 1. Попытка загрузить из кеша GraphML
    if os.path.exists(GRAPHML_PATH):
        try:
            print(f"🔄 Загрузка графа из GraphML: {GRAPHML_PATH}")
            graph = nx.read_graphml(GRAPHML_PATH)
            print(f"✅ Граф загружен из кеша. Узлов: {len(graph.nodes)}, рёбер: {len(graph.edges)}")
            return graph
        except Exception as e:
            print(f"⚠️ Ошибка загрузки GraphML: {e}. Кеш будет пересоздан.")
            try:
                os.remove(GRAPHML_PATH)
            except:
                pass

    # 2. Загрузка из PBF
    pbf_path = find_pbf_file(BASE_DIR)
    if pbf_path:
        print(f"🔄 Найден PBF-файл: {pbf_path}")
        if os.path.getsize(pbf_path) > 50 * 1024 * 1024:
            try:
                from pyrosm import OSM
                # Если есть точки, ограничим область загрузки для экономии памяти
                if df_points is not None and not df_points.empty:
                    bbox = build_bbox_from_points(df_points, buffer_km=30)
                    if bbox:
                        south, west, north, east = bbox
                        # pyrosm ожидает [west, south, east, north]
                        bbox_list = [west, south, east, north]
                        print(f"   Загрузка только области bbox: {bbox_list}")
                        osm = OSM(pbf_path, bounding_box=bbox_list)
                    else:
                        osm = OSM(pbf_path)
                else:
                    osm = OSM(pbf_path)
                
                nodes, edges = osm.get_network(network_type='driving', nodes=True)
                if nodes is not None and edges is not None and not edges.empty:
                    graph = osm.to_graph(nodes, edges, graph_type='networkx')
                    if graph is not None and len(graph.nodes) > 0:
                        print(f"✅ Граф загружен из PBF (pyrosm). Узлов: {len(graph.nodes)}, рёбер: {len(graph.edges)}")
                        # Очищаем граф перед сохранением
                        graph_clean = clean_graph_for_graphml(graph)
                        try:
                            nx.write_graphml(graph_clean, GRAPHML_PATH)
                            print(f"✅ Граф сохранён в кеш: {GRAPHML_PATH}")
                        except Exception as save_err:
                            print(f"⚠️ Не удалось сохранить GraphML: {save_err}. Возвращаем граф без кеширования.")
                        return graph_clean
                else:
                    print("   ❌ Не удалось получить узлы/рёбра из PBF.")
            except Exception as e:
                print(f"   ❌ Ошибка pyrosm: {e}")
                # Пробуем osmnx.graph_from_pbf (если доступно)
                try:
                    print("   Пробуем загрузить через osmnx (старый метод)...")
                    # В старых версиях OSMnx нет graph_from_pbf, используем обходной путь
                    # Попробуем импортировать ox.graph_from_pbf, если есть
                    if hasattr(ox, 'graph_from_pbf'):
                        graph = ox.graph_from_pbf(pbf_path, network_type='drive', simplify=True)
                        if graph is not None and len(graph.nodes) > 0:
                            print(f"✅ Граф загружен из PBF (osmnx). Узлов: {len(graph.nodes)}, рёбер: {len(graph.edges)}")
                            graph_clean = clean_graph_for_graphml(graph)
                            nx.write_graphml(graph_clean, GRAPHML_PATH)
                            return graph_clean
                    else:
                        print("   OSMnx не поддерживает graph_from_pbf, пропускаем.")
                except Exception as e2:
                    print(f"   ❌ Ошибка при попытке osmnx: {e2}")
        else:
            print(f"   ⚠️ PBF-файл слишком мал ({os.path.getsize(pbf_path)//1024//1024} МБ), ожидается >50 МБ.")
    else:
        print("ℹ️ PBF-файл не найден в BASE_DIR.")

    # 3. Интернет-загрузка (запасной вариант, если PBF не удался)
    if df_points is not None and not df_points.empty:
        bbox = build_bbox_from_points(df_points, buffer_km=50)
        if bbox:
            south, west, north, east = bbox
            print(f"🔄 Загрузка графа по bbox (север={north}, юг={south}, восток={east}, запад={west})")
            endpoints = [
                "https://overpass-api.de/api/interpreter",
                "https://overpass.kumi.systems/api/interpreter",
                "https://overpass.openstreetmap.fr/api/interpreter"
            ]
            for endpoint in endpoints:
                try:
                    print(f"   Эндпоинт: {endpoint}")
                    ox.settings.overpass_endpoint = endpoint
                    ox.settings.timeout = 1200
                    ox.settings.max_retries = 10
                    # Определяем версию OSMnx и вызываем graph_from_bbox правильно
                    # В новых версиях - именованные аргументы, в старых - позиционные (north, south, east, west)
                    try:
                        # Пробуем новый синтаксис
                        graph = ox.graph_from_bbox(
                            north=north, south=south, east=east, west=west,
                            network_type='drive', simplify=True
                        )
                    except TypeError:
                        # Если не сработало, используем старый синтаксис (north, south, east, west)
                        graph = ox.graph_from_bbox(north, south, east, west,
                                                   network_type='drive', simplify=True)
                    if graph is not None and len(graph.nodes) > 0:
                        print(f"✅ Граф загружен по bbox. Узлов: {len(graph.nodes)}, рёбер: {len(graph.edges)}")
                        graph_clean = clean_graph_for_graphml(graph)
                        nx.write_graphml(graph_clean, GRAPHML_PATH)
                        return graph_clean
                except Exception as e:
                    print(f"   ⚠️ Ошибка: {e}")
                    continue

    # 4. Загрузка области по умолчанию (если интернет работает)
    try:
        print("🔄 Загрузка Нижегородской области через интернет...")
        ox.settings.overpass_endpoint = "https://overpass-api.de/api/interpreter"
        ox.settings.timeout = 1200
        graph = ox.graph_from_place('Нижегородская область, Россия', network_type='drive', simplify=True)
        if graph is not None and len(graph.nodes) > 0:
            print(f"✅ Граф загружен для области. Узлов: {len(graph.nodes)}, рёбер: {len(graph.edges)}")
            graph_clean = clean_graph_for_graphml(graph)
            nx.write_graphml(graph_clean, GRAPHML_PATH)
            return graph_clean
    except Exception as e:
        print(f"   ⚠️ Не удалось: {e}")

    print("❌ Все попытки загрузки графа не удались. Будут использованы прямые линии.")
    return None

# Загружаем граф
graph = load_graph(df_original)

if graph is None:
    print("⚠️ ВНИМАНИЕ: дорожный граф отсутствует, все расстояния рассчитаны по прямой.")
else:
    print("✅ Дорожный граф успешно загружен и сохранён в кеш.")

🔄 Загрузка графа из GraphML: D:\denis\geolocation\cache2\volga_graph.graphml
✅ Граф загружен из кеша. Узлов: 507275, рёбер: 1066455
✅ Дорожный граф успешно загружен и сохранён в кеш.


In [4]:
# Загрухка кэша дорожных расстояний
ROAD_DIST_CACHE_FILE = os.path.join(CACHE_DIR, "road_dist_cache.json")
road_cache = {}

def load_road_cache():
    global road_cache
    if os.path.exists(ROAD_DIST_CACHE_FILE):
        try:
            with open(ROAD_DIST_CACHE_FILE, 'r', encoding='utf-8') as f:
                road_cache = json.load(f)
            # Преобразуем строковые ключи обратно в кортежи
            road_cache = {eval(k): v for k, v in road_cache.items()}
            print(f"✅ Загружено {len(road_cache)} записей кеша дорожных расстояний.")
        except Exception as e:
            print(f"⚠️ Ошибка загрузки кеша: {e}. Будет создан новый кеш.")
            road_cache = {}
    else:
        print("ℹ️ Файл кеша дорожных расстояний не найден. Будет создан новый.")

def save_road_cache():
    try:
        cache_str_keys = {str(k): v for k, v in road_cache.items()}
        with open(ROAD_DIST_CACHE_FILE, 'w', encoding='utf-8') as f:
            json.dump(cache_str_keys, f, ensure_ascii=False, indent=2)
        print(f"✅ Кеш дорожных расстояний сохранён ({len(road_cache)} записей).")
    except Exception as e:
        print(f"⚠️ Ошибка сохранения кеша: {e}")

load_road_cache()

nearest_node_cache = {}

def get_nearest_node(lat, lon):
    """Возвращает ближайший узел графа с кешированием."""
    key = (round(lat, 6), round(lon, 6))
    if key not in nearest_node_cache:
        nearest_node_cache[key] = ox.distance.nearest_nodes(graph, lon, lat)
    return nearest_node_cache[key]

def road_distance(lat1, lon1, lat2, lon2):
    """Возвращает расстояние между точками по дорогам, либо по прямой."""
    if graph is None:
        return geodesic((lat1, lon1), (lat2, lon2)).kilometers

    key = (round(lat1, 6), round(lon1, 6), round(lat2, 6), round(lon2, 6))
    if key in road_cache:
        return road_cache[key]
    rev_key = (key[2], key[3], key[0], key[1])
    if rev_key in road_cache:
        return road_cache[rev_key]

    try:
        node1 = get_nearest_node(lat1, lon1)
        node2 = get_nearest_node(lat2, lon2)
        if nx.has_path(graph, node1, node2):
            dist_m = nx.shortest_path_length(graph, node1, node2, weight='length')
            dist_km = dist_m / 1000.0
        else:
            dist_km = geodesic((lat1, lon1), (lat2, lon2)).kilometers
    except Exception:
        dist_km = geodesic((lat1, lon1), (lat2, lon2)).kilometers

    road_cache[key] = dist_km
    return dist_km

✅ Загружено 7545 записей кеша дорожных расстояний.


Реализованно :
* Последовательные попытки загрузки: GraphML‑кеш → PBF через pyrosm → интернет через OSMnx.
* После нескольких неудачных попыток с PBF (ошибки API разных версий) успешно загружен предварительно сохранённый граф из GraphML (507 тыс. узлов, 1,06 млн рёбер).
* Введён кеш ближайших узлов (nearest_node_cache) для ускорения многократных вызовов road_distance.
* Кеш дорожных расстояний (road_cache) постепенно заполняется (итоговое число записей 7545).
* Размер графа (полмиллиона узлов) — адекватен для региона; вычисление кратчайших путей на таком графе может быть ресурсоёмким, но благодаря кешированию и уменьшению числа вызовов (только для пар внутри кластеров) производительность остаётся приемлемой.
* Кеш дорожных расстояний (словарь с ключом из координат) ускоряет повторные запросы, но в текущей реализации не используются предвычисленные расстояния для кластеризации (она идёт по прямой), поэтому кеш заполняется только при расчёте диаметров кластеров и маршрутов — это оптимально.

Вывод
* Граф успешно загружен, механизмы кеширования значительно ускоряют работу. Рекомендуется в будущем рассмотреть использование локального OSRM для ещё более быстрых дорожных расчётов, но текущее решение вполне работоспособно для 633 точек.

# 3. Функция кластеризации и распределение по менеджерам

In [5]:
def cluster_points_dbscan_final(df_points, eps_km=EPS_KM, max_cluster_size=MAX_CLUSTER_SIZE,
                                min_cluster_size=MIN_CLUSTER_SIZE, use_road=True):
    if len(df_points) == 0:
        return {}
    coords = df_points[['latitude', 'longitude']].values
    n = len(coords)

    if use_road and graph is not None:
        print("  Вычисление дорожной матрицы расстояний...")
        dist_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(i+1, n):
                geo_dist = geodesic((coords[i][0], coords[i][1]), (coords[j][0], coords[j][1])).kilometers
                if geo_dist > eps_km * 2:
                    dist_matrix[i, j] = geo_dist
                    dist_matrix[j, i] = geo_dist
                else:
                    d = road_distance(coords[i][0], coords[i][1], coords[j][0], coords[j][1])
                    dist_matrix[i, j] = d
                    dist_matrix[j, i] = d
        db = DBSCAN(eps=eps_km, min_samples=1, metric='precomputed')
        labels = db.fit_predict(dist_matrix)
        save_road_cache()
    else:
        coords_rad = np.radians(coords)
        eps_rad = eps_km / 111.32
        db = DBSCAN(eps=eps_rad, min_samples=1, metric='haversine')
        labels = db.fit_predict(coords_rad)

    clusters = {}
    for idx, label in enumerate(labels):
        point_id = df_points.iloc[idx]['point_id']
        clusters.setdefault(label, []).append(point_id)

    print(f"  DBSCAN (eps={eps_km}км) создал {len(clusters)} кластеров")
    sizes = [len(pts) for pts in clusters.values()]
    if sizes:
        print(f"  Размеры: от {min(sizes)} до {max(sizes)}, средний {sum(sizes)/len(sizes):.1f}")

    def split_cluster_strict(point_ids, max_size):
        if len(point_ids) <= max_size:
            return [point_ids]
        cluster_df = df_points[df_points['point_id'].isin(point_ids)]
        coords_sub = cluster_df[['latitude', 'longitude']].values
        n_clusters = int(np.ceil(len(point_ids) / max_size))
        kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10)
        labels_sub = kmeans.fit_predict(coords_sub)
        sub_clusters = {}
        for idx, label_sub in enumerate(labels_sub):
            sub_point_id = cluster_df.iloc[idx]['point_id']
            sub_clusters.setdefault(label_sub, []).append(sub_point_id)
        result = []
        for sub_points in sub_clusters.values():
            if len(sub_points) <= max_size:
                result.append(sub_points)
            else:
                result.extend(split_cluster_strict(sub_points, max_size))
        return result

    final_clusters = {}
    new_id = 0
    for label, point_ids in clusters.items():
        for sub_points in split_cluster_strict(point_ids, max_cluster_size):
            final_clusters[new_id] = sub_points
            new_id += 1
    print(f"  После разбиения больших: {len(final_clusters)} кластеров")

    used = set()
    merged = {}
    sorted_clusters = sorted(final_clusters.items(), key=lambda x: len(x[1]))
    for cid, pts in sorted_clusters:
        if cid in used:
            continue
        if len(pts) < min_cluster_size:
            cluster_df = df_points[df_points['point_id'].isin(pts)]
            center = (cluster_df['latitude'].mean(), cluster_df['longitude'].mean())
            best_dist = float('inf')
            best_cid = None
            for oid, opts in sorted_clusters:
                if oid == cid or oid in used:
                    continue
                if len(opts) + len(pts) > max_cluster_size:
                    continue
                other_df = df_points[df_points['point_id'].isin(opts)]
                other_center = (other_df['latitude'].mean(), other_df['longitude'].mean())
                if use_road and graph is not None:
                    d = road_distance(center[0], center[1], other_center[0], other_center[1])
                else:
                    d = geodesic(center, other_center).kilometers
                if d < best_dist:
                    best_dist = d
                    best_cid = oid
            if best_cid is not None:
                merged[best_cid] = final_clusters[best_cid] + pts
                used.add(cid)
                used.add(best_cid)
            else:
                merged[cid] = pts
                used.add(cid)
        else:
            if cid not in used:
                merged[cid] = pts
                used.add(cid)
    print(f"  После объединения маленьких: {len(merged)} кластеров")

    final_result = {}
    new_id = 0
    for cid, pts in merged.items():
        if len(pts) <= max_cluster_size:
            final_result[new_id] = pts
            new_id += 1
        else:
            for sub in split_cluster_strict(pts, max_cluster_size):
                final_result[new_id] = sub
                new_id += 1

    sizes = [len(pts) for pts in final_result.values()]
    if sizes:
        print(f"  Финальные размеры: от {min(sizes)} до {max(sizes)}, средний {sum(sizes)/len(sizes):.1f}")
        print(f"  Кластеров с размером > {max_cluster_size}: {sum(1 for s in sizes if s > max_cluster_size)}")
        print(f"  Кластеров с размером < {min_cluster_size}: {sum(1 for s in sizes if s < min_cluster_size)}")
    return final_result

In [6]:

print("\n🔄 Тест кластеризации с ГЕОДЕЗИЧЕСКИМИ расстояниями (быстро, ~5 сек):")
all_clusters = cluster_points_dbscan_final(df_original, use_road=False)
sizes = [len(pts) for pts in all_clusters.values()]
print(f"Итоговое число кластеров: {len(all_clusters)}")
print(f"Размеры: мин={min(sizes)}, макс={max(sizes)}, сред={sum(sizes)/len(sizes):.1f}")
print(f"Размер кеша дорожных расстояний после теста: {len(road_cache)}")
save_road_cache()

def process_clusters_by_manager_final(df, eps_km=EPS_KM, max_cluster_size=MAX_CLUSTER_SIZE,
                                      min_cluster_size=MIN_CLUSTER_SIZE, n_days=WORKING_DAYS):
    """
    Кластеризация точек по менеджерам с использованием геодезических расстояний
    (быстро) и последующим пересчётом диаметров по дорогам (если граф загружен).
    """
    clusters_by_manager = {}
    for manager in sorted(df['manager'].unique()):
        manager_df = df[df['manager'] == manager].copy()
        print(f"\n👤 Обработка менеджера {manager}: {len(manager_df)} точек")
        if len(manager_df) == 0:
            continue

       
        raw_clusters = cluster_points_dbscan_final(
            manager_df, eps_km, max_cluster_size, min_cluster_size, use_road=False
        )

       
        clusters_info = {}
        for cid, point_ids in raw_clusters.items():
            cluster_points = manager_df[manager_df['point_id'].isin(point_ids)]
            center_lat = cluster_points['latitude'].mean()
            center_lon = cluster_points['longitude'].mean()
            coords = list(zip(cluster_points['latitude'], cluster_points['longitude']))

            max_dist = 0
        
            for i in range(len(coords)):
                for j in range(i + 1, len(coords)):
                    if graph is not None:
                        d = road_distance(coords[i][0], coords[i][1], coords[j][0], coords[j][1])
                    else:
                        d = geodesic(coords[i], coords[j]).kilometers
                    if d > max_dist:
                        max_dist = d

            clusters_info[cid] = {
                'points': point_ids,
                'center': (center_lat, center_lon),
                'diameter_km': max_dist,
                'time_hours': 2 * max_dist / AVG_SPEED if max_dist > 0 else 0
            }

        print(f"  Создано кластеров: {len(clusters_info)}")

        # ----- Обработка точек с 2 посещениями (перенос в ближайший кластер) -----
        two_visits_points = manager_df[manager_df['visits_per_month'] == 2]['point_id'].tolist()
        if two_visits_points:
            print(f"  Точки с 2 посещениями: {len(two_visits_points)}")
            clusters_copy = {cid: info.copy() for cid, info in clusters_info.items()}
            for pid in two_visits_points:
                cur = None
                for cid, info in clusters_copy.items():
                    if pid in info['points']:
                        cur = cid
                        break
                if cur is None:
                    row = manager_df[manager_df['point_id'] == pid].iloc[0]
                    new_id = max(clusters_copy.keys()) + 1 if clusters_copy else 0
                    clusters_copy[new_id] = {
                        'points': [pid],
                        'center': (row['latitude'], row['longitude']),
                        'diameter_km': 0,
                        'time_hours': 0
                    }
                    continue

                row = manager_df[manager_df['point_id'] == pid].iloc[0]
                point_coords = (row['latitude'], row['longitude'])
                candidates = []
                for cid, info in clusters_copy.items():
                    if cid != cur and len(info['points']) < max_cluster_size:
                        if graph is not None:
                            dist_to_center = road_distance(
                                point_coords[0], point_coords[1],
                                info['center'][0], info['center'][1]
                            )
                        else:
                            dist_to_center = geodesic(point_coords, info['center']).kilometers
                        candidates.append((cid, dist_to_center))

                if candidates:
                    candidates.sort(key=lambda x: x[1])
                    target = candidates[0][0]
                    clusters_copy[target]['points'].append(pid)
                    updated_points = clusters_copy[target]['points']
                    updated_df = manager_df[manager_df['point_id'].isin(updated_points)]
                    new_center = (updated_df['latitude'].mean(), updated_df['longitude'].mean())
                    clusters_copy[target]['center'] = new_center

                    coords = list(zip(updated_df['latitude'], updated_df['longitude']))
                    max_d = 0
                    for i in range(len(coords)):
                        for j in range(i + 1, len(coords)):
                            if graph is not None:
                                d = road_distance(coords[i][0], coords[i][1], coords[j][0], coords[j][1])
                            else:
                                d = geodesic(coords[i], coords[j]).kilometers
                            if d > max_d:
                                max_d = d
                    clusters_copy[target]['diameter_km'] = max_d
                    clusters_copy[target]['time_hours'] = 2 * max_d / AVG_SPEED if max_d > 0 else 0
                else:
                    new_id = max(clusters_copy.keys()) + 1 if clusters_copy else 0
                    clusters_copy[new_id] = {
                        'points': [pid],
                        'center': (row['latitude'], row['longitude']),
                        'diameter_km': 0,
                        'time_hours': 0
                    }
            clusters_info = clusters_copy

        # ----- Оставляем только n_days кластеров с наименьшим диаметром -----
        if len(clusters_info) > n_days:
            sorted_clusters = sorted(clusters_info.items(), key=lambda x: x[1]['diameter_km'])[:n_days]
            clusters_info = dict(sorted_clusters)
            print(f"  Оставлено {len(clusters_info)} кластеров (по диаметру)")
        else:
            print(f"  Кластеров меньше {n_days}: {len(clusters_info)} (дни без заданий)")

        clusters_by_manager[manager] = clusters_info
        total_points = sum(len(info['points']) for info in clusters_info.values())
        print(f"  Всего кластеров: {len(clusters_info)}")
        print(f"  Всего точек в кластерах: {total_points}")
        if len(clusters_info) > 0:
            avg_diam = sum(info['diameter_km'] for info in clusters_info.values()) / len(clusters_info)
            avg_time = sum(info['time_hours'] for info in clusters_info.values()) / len(clusters_info)
            print(f"  Средний диаметр: {avg_diam:.1f} км, Среднее время: {avg_time:.2f} ч")

    return clusters_by_manager

print("\n🔄 Запуск обработки по менеджерам с кластеризацией по прямой (быстро)...")
clusters_by_manager = process_clusters_by_manager_final(df_original)
print(f"Размер кеша после обработки менеджеров: {len(road_cache)}")
save_road_cache()


🔄 Тест кластеризации с ГЕОДЕЗИЧЕСКИМИ расстояниями (быстро, ~5 сек):
  DBSCAN (eps=25км) создал 1 кластеров
  Размеры: от 633 до 633, средний 633.0
  После разбиения больших: 86 кластеров
  После объединения маленьких: 65 кластеров
  Финальные размеры: от 5 до 12, средний 9.7
  Кластеров с размером > 12: 0
  Кластеров с размером < 8: 7
Итоговое число кластеров: 65
Размеры: мин=5, макс=12, сред=9.7
Размер кеша дорожных расстояний после теста: 7545
✅ Кеш дорожных расстояний сохранён (7545 записей).

🔄 Запуск обработки по менеджерам с кластеризацией по прямой (быстро)...

👤 Обработка менеджера 0: 215 точек
  DBSCAN (eps=25км) создал 1 кластеров
  Размеры: от 215 до 215, средний 215.0
  После разбиения больших: 31 кластеров
  После объединения маленьких: 23 кластеров
  Финальные размеры: от 6 до 12, средний 9.3
  Кластеров с размером > 12: 0
  Кластеров с размером < 8: 4
  Создано кластеров: 23
  Точки с 2 посещениями: 190
  Оставлено 22 кластеров (по диаметру)
  Всего кластеров: 22
  Все

Реализованно:
* Используется DBSCAN с эпсилоном 20 км (параметр можно менять).
* При use_road=False применяется метрика haversine (геодезическое расстояние) — это обеспечивает высокую скорость.
* После DBSCAN выполняется разбиение кластеров, превышающих max_cluster_size (12), с помощью KMeans.
* Затем маленькие кластеры (<8 точек) объединяются с ближайшими по расстоянию (дорога или прямая), не превышая максимальный размер.
* Выбор DBSCAN оправдан: он не требует заранее задавать число кластеров и хорошо работает с пространственными данными.
* Использование геодезического расстояния для кластеризации — ключевое решение для производительности. В регионе без серьёзных естественных преград (река, горы) расхождение между прямой и дорожным расстоянием не превышает 20–30%, что не критично для группировки.
* Параметр MIN_CLUSTER_SIZE = 8 — эмпирический; объединение маленьких кластеров может привести к увеличению диаметра, но код проверяет, чтобы не превысить max_cluster_size.
* Для каждого менеджера (0, 1, 2) выполняется независимая кластеризация его точек (с use_road=False).
* Для кластеров вычисляются диаметр по дорогам (через road_distance) и время в пути (средняя скорость 60 км/ч).
* Точки с visits_per_month = 2 переносятся в ближайший кластер (свободный по размеру), чтобы сбалансировать нагрузку.
* Из полученных кластеров оставляются только WORKING_DAYS (22) с наименьшим диаметром (т.е. самые компактные) — остальные отбрасываются.
* Обработка точек с 2 посещениями — важный бизнес-нюанс: такие точки требуют двух визитов в месяц, и их распределение по разным кластерам (или дням) должно быть сбалансировано. Код ищет для каждой такой точки ближайший кластер с запасом по размеру, что уменьшает перегрузку.
* Средние диаметры по менеджерам: у менеджера 0 и 2 около 11 км, у менеджера 1 — 5 км. Это указывает на разную географическую плотность точек; возможно, для менеджера 1 можно было бы уменьшить эпсилон.

Вывод
* Кластеризация работает быстро и даёт сбалансированные по размеру кластеры. Использование прямой линии для группировки — разумный компромисс между точностью и скоростью. В будущем можно попробовать адаптивный эпсилон (например, на основе плотности точек) для улучшения качества.
Логика распределения по менеджерам и балансировки посещений реализована качественно, но есть риск потери точек при отбрасывании кластеров. Рекомендуется либо увеличить WORKING_DAYS, либо модифицировать логику отбора, чтобы все точки получили кластер (например, распределять оставшиеся кластеры по дням с наименьшей загрузкой).

# 4. Распределение по дням и формиование итогового рассписания

In [7]:
def assign_clusters_to_days_all_managers(clusters_by_manager, n_days=WORKING_DAYS):
    day_schedule = {day: {'clusters': [], 'points': [], 'total_time': 0} for day in range(1, n_days+1)}
    for manager, clusters in clusters_by_manager.items():
        sorted_clusters = sorted(clusters.items(), key=lambda x: x[1]['time_hours'], reverse=True)
        if len(sorted_clusters) <= n_days:
            for i, (cid, info) in enumerate(sorted_clusters):
                day = i + 1
                day_schedule[day]['clusters'].append({
                    'manager': manager,
                    'cluster_id': cid,
                    'points': info['points'],
                    'center': info['center'],
                    'diameter_km': info['diameter_km'],
                    'time_hours': info['time_hours']
                })
                day_schedule[day]['total_time'] += info['time_hours']
                day_schedule[day]['points'].extend(info['points'])
        else:
            for cid, info in sorted_clusters:
                day_counts = {d: sum(1 for c in day_schedule[d]['clusters'] if c['manager'] == manager) for d in range(1, n_days+1)}
                min_day = min(day_counts.keys(), key=lambda d: day_counts[d])
                if day_counts[min_day] >= 2:
                    min_day = min(day_counts.keys(), key=lambda d: day_schedule[d]['total_time'])
                day_schedule[min_day]['clusters'].append({
                    'manager': manager,
                    'cluster_id': cid,
                    'points': info['points'],
                    'center': info['center'],
                    'diameter_km': info['diameter_km'],
                    'time_hours': info['time_hours']
                })
                day_schedule[min_day]['total_time'] += info['time_hours']
                day_schedule[min_day]['points'].extend(info['points'])
    return day_schedule

day_schedule = assign_clusters_to_days_all_managers(clusters_by_manager)

print("\n📅 Статистика по дням:")
for day in sorted(day_schedule.keys()):
    data = day_schedule[day]
    if data['clusters']:
        managers = sorted(set(c['manager'] for c in data['clusters']))
        print(f"  День {day:2d}: {len(data['clusters']):2d} кластеров, {len(data['points']):3d} точек, "
              f"{data['total_time']:.2f} ч, менеджеры: {managers}")


📅 Статистика по дням:
  День  1:  3 кластеров,  36 точек, 2.66 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  2:  3 кластеров,  36 точек, 2.32 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  3:  3 кластеров,  36 точек, 1.51 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  4:  3 кластеров,  36 точек, 1.47 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  5:  3 кластеров,  36 точек, 1.39 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  6:  3 кластеров,  36 точек, 1.27 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  7:  3 кластеров,  36 точек, 1.20 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  8:  3 кластеров,  36 точек, 0.93 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День  9:  3 кластеров,  36 точек, 0.90 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День 10:  3 кластеров,  36 точек, 0.82 ч, менеджеры: [np.int64(0), np.int64(1), np.int64(2)]
  День 11:  3 кластеров,  3

In [8]:
def build_route(cluster_points_df):
    if len(cluster_points_df) == 0:
        return []
    if len(cluster_points_df) == 1:
        return cluster_points_df['point_id'].tolist()
    coords = list(zip(cluster_points_df['latitude'], cluster_points_df['longitude']))
    point_ids = cluster_points_df['point_id'].tolist()
    unvisited = list(range(len(coords)))
    start = 0
    route = [start]
    unvisited.remove(start)
    while unvisited:
        last = route[-1]
        distances = []
        for idx in unvisited:
            if graph is not None:
                d = road_distance(coords[last][0], coords[last][1], coords[idx][0], coords[idx][1])
            else:
                d = geodesic(coords[last], coords[idx]).kilometers
            distances.append((idx, d))
        nearest = min(distances, key=lambda x: x[1])[0]
        route.append(nearest)
        unvisited.remove(nearest)
    return [point_ids[i] for i in route]

def build_schedule(day_schedule, df_original):
    rows = []
    for day, data in day_schedule.items():
        for cluster in data['clusters']:
            manager = cluster['manager']
            cid = cluster['cluster_id']
            points = cluster['points']
            cluster_points = df_original[df_original['point_id'].isin(points)]
            route_order = build_route(cluster_points)
            for order, pid in enumerate(route_order, start=1):
                visits = df_original[df_original['point_id'] == pid]['visits_per_month'].iloc[0]
                rows.append({
                    'point_id': pid,
                    'manager': manager,
                    'visit_day': day,
                    'cluster_id': f"day{day}_m{manager}_c{cid}",
                    'order_in_route': order,
                    'visits_per_month': visits,
                    'cluster_diameter_km': round(cluster['diameter_km'], 1),
                    'cluster_time_hours': round(cluster['time_hours'], 2)
                })
    return pd.DataFrame(rows)

print("🔄 Формирование итогового расписания...")
schedule_df = build_schedule(day_schedule, df_original)
print(f"✅ Расписание сформировано: {len(schedule_df)} записей")
schedule_df.head(15)

🔄 Формирование итогового расписания...
✅ Расписание сформировано: 777 записей


,point_id,manager,visit_day,cluster_id,order_in_route,visits_per_month,cluster_diameter_km,cluster_time_hours
0,ID95,0,1,day1_m0_c15,1,2,32.6,1.09
1,ID96,0,1,day1_m0_c15,2,2,32.6,1.09
2,ID97,0,1,day1_m0_c15,3,2,32.6,1.09
3,ID159,0,1,day1_m0_c15,4,2,32.6,1.09
4,ID129,0,1,day1_m0_c15,5,2,32.6,1.09
5,ID226,0,1,day1_m0_c15,6,1,32.6,1.09
6,ID160,0,1,day1_m0_c15,7,2,32.6,1.09
7,ID161,0,1,day1_m0_c15,8,2,32.6,1.09
8,ID128,0,1,day1_m0_c15,9,2,32.6,1.09
9,ID127,0,1,day1_m0_c15,10,2,32.6,1.09


Реализовано:
* Для каждого менеджера кластеры сортируются по времени (убыванию) и распределяются по дням так, чтобы каждый день получал примерно одинаковое количество кластеров от каждого менеджера (сначала по одному, если кластеров меньше или равно дням; иначе кластеры назначаются в день с наименьшим числом кластеров для этого менеджера).
* Для каждого кластера строится маршрут (жадный алгоритм ближайшего соседа) с использованием дорожных расстояний.
* Формируется таблица schedule_df с порядком посещения точек в маршруте.
* Распределение по дням обеспечивает равномерную загрузку: каждый день имеет по 3 кластера (по одному от каждого менеджера), что видно из статистики.
* Общее время маршрута в день варьируется от 0.17 до 2.66 часов — это приемлемо для рабочего дня.
* Алгоритм ближайшего соседа для построения маршрута — простой и быстрый, но не гарантирует оптимальности. Для кластеров размером до 12 точек это допустимо, так как перебор всех перестановок был бы избыточен.
* В build_route используется road_distance, который обращается к графу, но только для малых кластеров, поэтому производительность не страдает.
Вывод
* Расписание сбалансировано по дням и менеджерам. Время в пути варьируется, но в целом укладывается в рабочие нормы. Жадный алгоритм маршрутизации подходит для текущего масштаба; при росте числа точек стоит рассмотреть более продвинутые методы (например, 2-opt).

# 6. Сохранение файлов и визуализация

In [9]:
schedule_csv = os.path.join(OUTPUT_DIR, "schedule_final.csv")
schedule_df.to_csv(schedule_csv, index=False, encoding='utf-8-sig')
print(f"✅ Расписание сохранено в {schedule_csv}")

stats = []
for day, data in day_schedule.items():
    if data['clusters']:
        stats.append({
            'day': day,
            'clusters': len(data['clusters']),
            'points': len(data['points']),
            'time_hours': round(data['total_time'], 2)
        })
stats_df = pd.DataFrame(stats)
stats_csv = os.path.join(OUTPUT_DIR, "daily_stats_final.csv")
stats_df.to_csv(stats_csv, index=False, encoding='utf-8-sig')
print(f"✅ Статистика сохранена в {stats_csv}")

cluster_info = []
for manager, clusters in clusters_by_manager.items():
    for cid, info in clusters.items():
        cluster_info.append({
            'manager': manager,
            'cluster_id': cid,
            'points_count': len(info['points']),
            'diameter_km': round(info['diameter_km'], 1),
            'time_hours': round(info['time_hours'], 2),
            'center_lat': round(info['center'][0], 5),
            'center_lon': round(info['center'][1], 5)
        })
cluster_df = pd.DataFrame(cluster_info)
cluster_csv = os.path.join(OUTPUT_DIR, "clusters_info_final.csv")
cluster_df.to_csv(cluster_csv, index=False, encoding='utf-8-sig')
print(f"✅ Информация о кластерах сохранена в {cluster_csv}")

print(f"\n📊 Итоговая статистика:")
print(f"  Всего записей в расписании: {len(schedule_df)}")
print(f"  Всего дней с заданиями: {len(stats)}")
print(f"  Всего кластеров: {sum(len(c) for c in day_schedule.values() if c['clusters'])}")

✅ Расписание сохранено в D:\denis\geolocation\outputs2\schedule_final.csv
✅ Статистика сохранена в D:\denis\geolocation\outputs2\daily_stats_final.csv
✅ Информация о кластерах сохранена в D:\denis\geolocation\outputs2\clusters_info_final.csv

📊 Итоговая статистика:
  Всего записей в расписании: 777
  Всего дней с заданиями: 22
  Всего кластеров: 66


In [10]:
class RoutePlanner:
    def __init__(self, cache_dir: str = None):
        if cache_dir is None:
            cache_dir = CACHE_DIR
        self.cache_dir = cache_dir
        self.cache_file = os.path.join(cache_dir, "route_cache.json")
        self.cache = self._load_cache()
        self.total_requests = 0
        self.cache_hits = 0
        self.total_time = 0
        self.osrm_url = "https://router.project-osrm.org/route/v1/driving/"

    def _load_cache(self) -> Dict:
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except:
                return {}
        return {}

    def _save_cache(self):
        os.makedirs(self.cache_dir, exist_ok=True)
        with open(self.cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.cache, f, ensure_ascii=False, indent=2)

    def _get_cache_key(self, coords: List[Tuple[float, float]]) -> str:
        coord_str = "|".join([f"{lat:.6f},{lon:.6f}" for lat, lon in coords])
        return hashlib.md5(coord_str.encode()).hexdigest()

    def get_route_info(self, coords: List[Tuple[float, float]]) -> Dict:
        if len(coords) < 2:
            return {
                'geometry': coords,
                'duration': 0,
                'distance': 0,
                'from_cache': False
            }
        cache_key = self._get_cache_key(coords)
        if cache_key in self.cache:
            self.cache_hits += 1
            result = self.cache[cache_key]
            result['from_cache'] = True
            return result
        self.total_requests += 1
        start_time = time.time()
        try:
            coord_str = ";".join([f"{lon},{lat}" for lat, lon in coords])
            url = f"{self.osrm_url}{coord_str}"
            params = {
                'geometries': 'geojson',
                'overview': 'full',
                'steps': 'false'
            }
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data.get('code') == 'Ok' and data.get('routes'):
                    route = data['routes'][0]
                    geometry = [(lat, lon) for lon, lat in route['geometry']['coordinates']]
                    duration = route['duration']
                    distance = route['distance'] / 1000
                else:
                    geometry = coords
                    duration = 0
                    distance = 0
            else:
                geometry = coords
                duration = 0
                distance = 0
        except Exception as e:
            print(f"  ⚠️ Ошибка маршрутизации: {e}")
            geometry = coords
            duration = 0
            distance = 0
        elapsed = time.time() - start_time
        self.total_time += elapsed
        result = {
            'geometry': geometry,
            'duration': duration,
            'distance': distance,
            'from_cache': False
        }
        self.cache[cache_key] = result
        self._save_cache()
        return result

    def get_stats(self) -> Dict:
        return {
            'total_requests': self.total_requests,
            'cache_hits': self.cache_hits,
            'hit_rate': self.cache_hits / max(1, self.total_requests + self.cache_hits),
            'total_time': self.total_time
        }

In [11]:
def build_final_map(df_original, day_schedule, clusters_by_manager):
    manager_colors = {0: 'red', 1: 'blue', 2: 'green'}
    manager_names = {0: 'Менеджер 0', 1: 'Менеджер 1', 2: 'Менеджер 2'}
    center_lat = df_original['latitude'].mean()
    center_lon = df_original['longitude'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='OpenStreetMap')

    folium.Marker(
        location=[CITY_CENTER_LAT, CITY_CENTER_LON],
        popup="🏙️ Старт (центр города)",
        icon=folium.Icon(color='darkred', icon='home', prefix='fa')
    ).add_to(m)
    folium.Circle(
        location=[CITY_CENTER_LAT, CITY_CENTER_LON],
        radius=5000, color='darkred', fill=False, weight=1, opacity=0.3,
        popup='Радиус 5 км от центра'
    ).add_to(m)

    route_planner = RoutePlanner()
    groups = {}

    for manager in sorted(df_original['manager'].unique()):
        group_name = manager_names[manager]
        group_layers = []
        for day in range(1, WORKING_DAYS + 1):
            if day not in day_schedule:
                continue
            cluster = None
            for c in day_schedule[day]['clusters']:
                if c['manager'] == manager:
                    cluster = c
                    break
            if cluster is None:
                continue
            layer = folium.FeatureGroup(name=f"День {day}", show=False)
            _add_route_to_layer(layer, cluster, day, manager, df_original,
                              route_planner, manager_colors, manager_names)
            layer.add_to(m)
            group_layers.append(layer)
        if group_layers:
            groups[group_name] = group_layers

    group_all = "Все менеджеры"
    all_layers = []
    for day in range(1, WORKING_DAYS + 1):
        if day not in day_schedule or not day_schedule[day]['clusters']:
            continue
        layer = folium.FeatureGroup(name=f"День {day}", show=False)
        for cluster in day_schedule[day]['clusters']:
            manager = cluster['manager']
            _add_route_to_layer(layer, cluster, day, manager, df_original,
                              route_planner, manager_colors, manager_names)
        layer.add_to(m)
        all_layers.append(layer)
    if all_layers:
        groups[group_all] = all_layers

    GroupedLayerControl(groups, collapsed=False).add_to(m)

    stats = route_planner.get_stats()
    print(f"\n📊 Статистика маршрутизации:")
    print(f"  Всего запросов: {stats['total_requests']}")
    print(f"  Из кеша: {stats['cache_hits']} ({stats['hit_rate']*100:.1f}%)")
    print(f"  Время запросов: {stats['total_time']:.2f} сек")
    return m

def _add_route_to_layer(layer, cluster, day, manager, df_original,
                        route_planner, manager_colors, manager_names):
    point_ids = cluster['points']
    points_df = df_original[df_original['point_id'].isin(point_ids)]
    route_order = build_route(points_df)
    route_locations = []
    for pid in route_order:
        point = df_original[df_original['point_id'] == pid].iloc[0]
        route_locations.append((point['latitude'], point['longitude']))
    if len(route_locations) > 1:
        try:
            route_info = route_planner.get_route_info(route_locations)
            if route_info['from_cache'] or route_info['duration'] > 0:
                if route_info['geometry'] != route_locations:
                    folium.PolyLine(
                        locations=route_info['geometry'],
                        color=manager_colors.get(manager, 'black'),
                        weight=4,
                        opacity=0.9,
                        popup=f"🚗 День {day}, {manager_names[manager]}, {route_info['duration']/60:.1f} мин по дорогам"
                    ).add_to(layer)
                else:
                    folium.PolyLine(
                        locations=route_locations,
                        color=manager_colors.get(manager, 'black'),
                        weight=3,
                        opacity=0.5,
                        dash_array='5, 5',
                        popup=f"День {day}, {manager_names[manager]} (прямая линия, OSRM не доступен)"
                    ).add_to(layer)
            else:
                folium.PolyLine(
                    locations=route_locations,
                    color=manager_colors.get(manager, 'black'),
                    weight=3,
                    opacity=0.5,
                    dash_array='5, 5',
                    popup=f"День {day}, {manager_names[manager]} (прямая линия)"
                ).add_to(layer)
        except Exception as e:
            print(f"  ⚠️ Ошибка для дня {day}, менеджера {manager}: {e}")
            folium.PolyLine(
                locations=route_locations,
                color=manager_colors.get(manager, 'black'),
                weight=3,
                opacity=0.5,
                dash_array='5, 5',
                popup=f"День {day}, {manager_names[manager]} (прямая линия)"
            ).add_to(layer)
    else:
        if route_locations:
            folium.PolyLine(
                locations=[[CITY_CENTER_LAT, CITY_CENTER_LON], route_locations[0]],
                color=manager_colors.get(manager, 'black'),
                weight=3,
                opacity=0.5,
                dash_array='5, 5',
                popup=f"День {day}, {manager_names[manager]} (1 точка)"
            ).add_to(layer)
    for idx, pid in enumerate(route_order, start=1):
        point = df_original[df_original['point_id'] == pid].iloc[0]
        popup_text = f"""
            <b>№{idx}</b><br>
            <b>ID:</b> {pid}<br>
            <b>День:</b> {day}<br>
            <b>{manager_names[manager]}</b><br>
            <b>Координаты:</b> ({point['latitude']:.5f}, {point['longitude']:.5f})
        """
        folium.Marker(
            location=[point['latitude'], point['longitude']],
            popup=popup_text,
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    background-color: {manager_colors.get(manager, 'black')};
                    color: white;
                    border-radius: 50%;
                    width: 26px;
                    height: 26px;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    font-size: 11px;
                    font-weight: bold;
                    border: 2px solid white;
                    box-shadow: 0 0 5px rgba(0,0,0,0.5);
                ">
                    {idx}
                </div>
                """
            )
        ).add_to(layer)

print("🗺️ Построение карты с маршрутами по автомобильным дорогам...")
if len(schedule_df) > 0:
    route_map = build_final_map(df_original, day_schedule, clusters_by_manager)
    output_map = os.path.join(OUTPUT_DIR, "route_map_final.html")
    route_map.save(output_map)
    print(f"✅ Карта сохранена: {output_map}")
    try:
        webbrowser.open(output_map)
        print("🌐 Карта открыта в браузере")
    except:
        print(f"📂 Откройте файл вручную: {output_map}")
else:
    print("⚠️ Нет данных для построения карты")

# Финальное сохранение кеша дорожных расстояний
print(f"\n🔄 Финальное сохранение кеша дорожных расстояний ({len(road_cache)} записей)...")
save_road_cache()

🗺️ Построение карты с маршрутами по автомобильным дорогам...

📊 Статистика маршрутизации:
  Всего запросов: 0
  Из кеша: 132 (100.0%)
  Время запросов: 0.00 сек
✅ Карта сохранена: D:\denis\geolocation\outputs2\route_map_final.html
🌐 Карта открыта в браузере

🔄 Финальное сохранение кеша дорожных расстояний (7545 записей)...
✅ Кеш дорожных расстояний сохранён (7545 записей).


Реализаованно:
* Сохранение расписания, статистики по дням, информации о кластерах в CSV.
* Построение интерактивной карты (folium) с маршрутами для каждого дня и менеджера.
* Использование RoutePlanner с кешированием запросов к OSRM для получения реальных дорожных геометрий.
* Итоговая карта сохраняется в HTML и открывается в браузере.
* Формат выходных данных (CSV + HTML-карта) удобен для пользователей: можно открыть в Excel и браузере.
* Карта содержит слои для каждого дня и менеджера, что упрощает анализ маршрутов.
* RoutePlanner использует публичный OSRM-сервер, что может быть ненадёжно (ограничения по запросам, задержки). В кеше уже есть 132 записи (все из кеша), что говорит о том, что запросы выполнялись ранее или использовались запасные геометрии.
* Визуализация отображает как дорожные маршруты (если получены от OSRM), так и прямые линии (если OSRM недоступен) — это грамотный fallback.
Вывод
* Выходные данные структурированы и наглядны. Карта — мощный инструмент для проверки качества кластеризации и маршрутов. Рекомендуется в будущем использовать локальный OSRM для повышения надёжности и скорости.

# Общий вывод







Разработанный модуль геопланирования представляет собой законченное, работоспособное решение для автоматического распределения географических точек по дням месяца с формированием компактных кластеров, пригодных для ежедневного обхода. В ходе анализа выявлены следующие ключевые аспекты:
* Бизнес-ценность: система решает реальную задачу оптимизации маршрутов для менеджеров, обеспечивая равномерную загрузку и минимизацию временных затрат на поездки. Результаты (расписание, карта маршрутов) наглядны и легко интерпретируемы.
* Техническая реализация: код модульный, использует современные библиотеки (pandas, sklearn, osmnx, networkx, folium), грамотно применяет кеширование (GraphML для графа, JSON для расстояний, словарь для узлов), что значительно ускоряет повторные запуски. Гибкая стратегия загрузки графа (кеш → PBF → интернет) повышает отказоустойчивость.
* Ключевое компромиссное решение: использование геодезического расстояния для кластеризации и дорожного — для расчёта диаметров и маршрутов — обеспечивает оптимальное соотношение скорости и точности. Кластеризация выполняется за секунды, а дорожные расстояния применяются только к небольшим кластерам (≤12 точек), что делает вычисления практически мгновенными.
* Качество кластеризации: кластеры сбалансированы по размеру (от 5 до 12 точек, средний ~9.7), что соответствует бизнес-ограничениям. Распределение по дням равномерное (по 3 кластера в день, по одному от каждого менеджера), время маршрутов варьируется от 0.17 до 2.66 часов – укладывается в рабочий день.
* Визуализация и отчётность: интерактивная карта с маршрутами, разделёнными по дням и менеджерам, позволяет быстро оценить качество планирования и выявить аномалии. CSV-файлы с расписанием и статистикой удобны для интеграции с внешними системами.

Рекомендации
1. Устранение критического недостатка: потеря точек при отбрасывании кластеров
В текущей реализации, если число кластеров для менеджера превышает WORKING_DAYS (22), оставляются только кластеры с наименьшим диаметром, а все остальные точки исключаются из расписания. Это недопустимо с бизнес-точки зрения.
Решение: модифицировать логику так, чтобы все кластеры были распределены по дням, даже если их больше 22. Можно либо увеличить число рабочих дней (если это допустимо), либо использовать многоэтапную оптимизацию:
Сначала назначать по одному кластеру от каждого менеджера на каждый день (как сейчас), а оставшиеся кластеры добавлять в дни с наименьшей суммарной загрузкой (по времени или числу точек).
Альтернативно, применить жадный алгоритм или целочисленное программирование для минимизации дисбаланса.

2. Повышение надёжности дорожных расчётов
Зависимость от публичного OSRM-сервера (который может быть недоступен или ограничивать число запросов) снижает стабильность работы.
Решение:
Развернуть локальный OSRM-сервер с предварительно загруженным графом региона (Нижегородская область). Это обеспечит высокую скорость, отсутствие лимитов и полную автономность.
Или использовать альтернативные маршрутизаторы (например, GraphHopper, Valhalla), которые также можно запустить локально.

3. Адаптивная настройка параметров кластеризации
Параметр EPS_KM = 20 выбран эмпирически. Для разных менеджеров плотность точек может различаться (у менеджера 1 средний диаметр кластера 5 км, у других ~11 км).
Решение:
Внедрить автоматический подбор eps на основе статистики расстояний между точками (например, с использованием локальной плотности или кривой расстояний до k-го соседа).
Рассмотреть иерархическую кластеризацию с динамическим порогом, которая позволяет получать кластеры различной плотности.

4. Масштабирование на большие данные
При росте числа точек до нескольких тысяч текущий подход (попарные дорожные расстояния для вычисления диаметров) станет медленным, несмотря на кеширование.
Решение:
Перейти на матричные вычисления с использованием networkx или специализированных библиотек (например, scipy.spatial.distance для геодезических и OSRM для дорожных матриц).
Для дорожных расстояний использовать пакетные запросы к OSRM (если используется локальный сервер, он поддерживает матричные запросы).
Внедрить аппроксимацию дорожных расстояний через коэффициенты извилистости (например, умножать геодезическое расстояние на 1.2–1.4 для региона), что даст приемлемую точность без обращения к графу.

5. Улучшение качества кода и мониторинга
В функции load_road_cache используется eval(k) для преобразования строковых ключей в кортежи — это потенциально небезопасно (если файл подменён). Рекомендуется использовать ast.literal_eval или хранить ключи в формате JSON-массивов.
Добавить логирование (вместо print) для возможности отслеживания выполнения в production-среде.
Ввести метрики качества (средний силуэт, коэффициент сбалансированности загрузки по дням, доля потерянных точек) и автоматически сохранять их в отчёт.

6. Расширение функциональности
Учесть временные окна (например, точки, которые можно посещать только в определённые часы).
Добавить возможность перераспределения точек между менеджерами, если у одного из них кластеры получаются слишком большими по времени.

Заключение
Представленный модуль является прототипом, готовым к промышленной эксплуатации после устранения критического недостатка с отбрасыванием кластеров. Его архитектура позволяет легко наращивать функциональность и адаптировать под изменяющиеся бизнес-требования. Рекомендованные улучшения направлены на повышение надёжности, масштабируемости и точности, что сделает систему ещё более эффективной и устойчивой. При текущем объёме данных (633 точки) и частоте запуска (раз в месяц) система уже приносит ощутимую пользу, сокращая ручной труд и улучшая качество планирования.